# Проверка only_excel клиентов в Озере (февраль)

Тетрадка:
1. Берет Excel за февраль.
2. Формирует `only_excel` относительно `sandbox_ai.stg__chesnov_aef__sa_acquiring_merchants` (активные на 1-е число месяца).
3. Проверяет `only_excel` в таблицах Озера и раскладывает по причинам:
   - нет в agreements;
   - `acq_class` не `SA`;
   - договор не активен на 1-е число месяца;
   - нет ТСП в merchants;
   - нет активных терминалов в феврале;
   - нет транзакций по DAG-периметру в феврале;
   - формально подходит, но отсутствует в snapshot merchants.

Подключение использует ваш формат (`user_name` + Kerberos).

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = "/home/jovyan/documents/Equaring/Проверки/отчет_февраль_2026.xlsx"
excel_agr_col = "ID договора"
excel_inn_col = "ИНН"
excel_contract_col = "Номер договора"

month_start = "2026-02-01"
month_end = "2026-02-29"

merchants_snapshot_table = "sandbox_ai.stg__chesnov_aef__sa_acquiring_merchants"

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

## Таблицы Озера, которые участвуют в проверке

- `ods_alpha.scd1_agreements`
- `ods_alpha.scd1_companies`
- `ods_alpha.scd1_merchants`
- `ods_alpha.scd1_pos_terminals`
- `ods_alpha.scd1_trx`
- `ods_alpha.scd1_trx_acq`
- `sandbox_ai.stg__chesnov_aef__sa_acquiring_merchants`

In [ ]:
def normalize_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    return s if re.fullmatch(r"\d+", s) else None

def normalize_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s

def to_sql_in_list(values):
    return ','.join([f"'{v}'" for v in values])

## 1) Формируем `only_excel` по `agr_id`

In [ ]:
df_excel = pd.read_excel(excel_path)

for col in [excel_agr_col, excel_inn_col, excel_contract_col]:
    if col not in df_excel.columns:
        raise ValueError(f"В Excel не найдена колонка: {col}")

excel_norm = pd.DataFrame({
    'agr_id': df_excel[excel_agr_col].apply(normalize_id),
    'inn': df_excel[excel_inn_col].apply(normalize_id),
    'c_agr_number': df_excel[excel_contract_col].apply(normalize_str),
}).dropna(how='all')

with imp:
    dm = imp.fetch(f"""
        select
            cast(agr_id as string) as agr_id,
            cast(inn as string) as inn,
            cast(c_agr_number as string) as c_agr_number,
            cast(d_valid_from as date) as d_valid_from,
            cast(d_valid_to as date) as d_valid_to
        from {merchants_snapshot_table}
        where cast(d_valid_from as date) <= cast('{month_start}' as date)
          and (
                d_valid_to is null
                or cast(d_valid_to as date) >= cast('{month_start}' as date)
              )
    """)

dm_norm = pd.DataFrame({
    'agr_id': dm['agr_id'].apply(normalize_id),
    'inn': dm['inn'].apply(normalize_id),
    'c_agr_number': dm['c_agr_number'].apply(normalize_str),
}).dropna(how='all')

set_excel_agr = set(excel_norm['agr_id'].dropna().tolist())
set_dm_agr = set(dm_norm['agr_id'].dropna().tolist())

only_excel_agr = sorted(list(set_excel_agr - set_dm_agr))
only_dm_agr = sorted(list(set_dm_agr - set_excel_agr))

print('excel_unique_agr =', len(set_excel_agr))
print('merchants_unique_agr =', len(set_dm_agr))
print('only_excel_agr_cnt =', len(only_excel_agr))
print('only_dm_agr_cnt =', len(only_dm_agr))

display(pd.DataFrame({'only_excel_agr_id': only_excel_agr[:100]}))

## 2) Проверяем `only_excel` в `agreements` и раскладываем причины (блок 1)

In [ ]:
def fetch_agr_base(agr_ids, chunk_size=500):
    chunks = []
    for i in range(0, len(agr_ids), chunk_size):
        sub = agr_ids[i:i+chunk_size]
        if not sub:
            continue
        in_list = to_sql_in_list(sub)
        sql = f"""
            select
                cast(a.abs_agr_id as string) as agr_id,
                a.acq_class,
                cast(a.d_valid_from as date) as d_valid_from,
                cast(a.d_valid_to as date) as d_valid_to,
                cast(a.n_agr as string) as n_agr,
                cast(a.n_cmp_client as string) as n_cmp_client,
                cast(a.c_agr_number as string) as c_agr_number,
                cast(c.c_inn as string) as inn
            from ods_alpha.scd1_agreements a
            left join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where cast(a.abs_agr_id as string) in ({in_list})
        """
        with imp:
            part = imp.fetch(sql)
        chunks.append(part)

    if not chunks:
        return pd.DataFrame(columns=['agr_id','acq_class','d_valid_from','d_valid_to','n_agr','n_cmp_client','c_agr_number','inn'])
    return pd.concat(chunks, ignore_index=True)

only_excel_df = pd.DataFrame({'agr_id': only_excel_agr})

if len(only_excel_agr) > 0:
    agr_base = fetch_agr_base(only_excel_agr)
    # Для причин достаточно одной строки на agr_id
    agr_base = agr_base.sort_values(['agr_id']).drop_duplicates(subset=['agr_id'], keep='first')
    only_excel_df = only_excel_df.merge(agr_base, on='agr_id', how='left')
else:
    for c in ['acq_class','d_valid_from','d_valid_to','n_agr','n_cmp_client','c_agr_number','inn']:
        only_excel_df[c] = None

only_excel_df['in_agreements'] = only_excel_df['acq_class'].notna()
only_excel_df['is_sa'] = only_excel_df['acq_class'].astype(str).eq('SA')
only_excel_df['active_on_month_start'] = (
    pd.to_datetime(only_excel_df['d_valid_from'], errors='coerce').dt.date <= pd.to_datetime(month_start).date()
) & (
    only_excel_df['d_valid_to'].isna() |
    (pd.to_datetime(only_excel_df['d_valid_to'], errors='coerce').dt.date >= pd.to_datetime(month_start).date())
)

display(only_excel_df.head(20))

## 3) Проверяем наличие ТСП/терминалов/транзакций для `only_excel` (блок 2)

Выполняется только для кандидатов, которые:
- есть в agreements;
- `acq_class = 'SA'`;
- активны на 1-е число месяца.

In [ ]:
candidates = only_excel_df[
    only_excel_df['in_agreements'] &
    only_excel_df['is_sa'] &
    only_excel_df['active_on_month_start']
].copy()

candidate_n_cmp = sorted([x for x in candidates['n_cmp_client'].dropna().astype(str).unique().tolist()])
candidate_n_agr = sorted([x for x in candidates['n_agr'].dropna().astype(str).unique().tolist()])

print('candidates_after_base_filters =', len(candidates))
print('candidate_n_cmp_cnt =', len(candidate_n_cmp))
print('candidate_n_agr_cnt =', len(candidate_n_agr))

merch_by_cmp = pd.DataFrame(columns=['n_cmp_client','has_retl'])
term_by_cmp = pd.DataFrame(columns=['n_cmp_client','has_active_term_feb'])
trx_by_agr = pd.DataFrame(columns=['n_agr','has_trx_feb_dag'])

if len(candidate_n_cmp) > 0:
    in_cmp = to_sql_in_list(candidate_n_cmp)
    with imp:
        merch_by_cmp = imp.fetch(f"""
            select
                cast(m.n_cmp as string) as n_cmp_client,
                1 as has_retl
            from ods_alpha.scd1_merchants m
            where cast(m.n_cmp as string) in ({in_cmp})
              and m.c_nmrc is not null
            group by cast(m.n_cmp as string)
        """)

        term_by_cmp = imp.fetch(f"""
            select
                cast(m.n_cmp as string) as n_cmp_client,
                1 as has_active_term_feb
            from ods_alpha.scd1_merchants m
            join ods_alpha.scd1_pos_terminals t
              on t.c_nmrc = m.c_nmrc
            where cast(m.n_cmp as string) in ({in_cmp})
              and t.c_nter is not null
              and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
              and (
                    t.d_ter_close is null
                    or cast(t.d_ter_close as date) >= cast('{month_start}' as date)
                  )
            group by cast(m.n_cmp as string)
        """)

if len(candidate_n_agr) > 0:
    # Чанкуем n_agr, чтобы не сделать слишком длинный IN
    trx_chunks = []
    step = 500
    for i in range(0, len(candidate_n_agr), step):
        sub = candidate_n_agr[i:i+step]
        in_agr = to_sql_in_list(sub)
        with imp:
            part = imp.fetch(f"""
                select
                    cast(ta.n_agr as string) as n_agr,
                    1 as has_trx_feb_dag
                from ods_alpha.scd1_trx_acq ta
                join ods_alpha.scd1_trx t
                  on t.n_trx = ta.n_trx
                where cast(ta.n_agr as string) in ({in_agr})
                  and t.c_trx_class = 'SA'
                  and t.c_trx_type = 'S01'
                  and t.c_nter is not null
                  and t.ods_deleted_flg <> '1'
                  and t.cf_trx_stat <> 'R'
                  and to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
                group by cast(ta.n_agr as string)
            """)
        trx_chunks.append(part)

    if trx_chunks:
        trx_by_agr = pd.concat(trx_chunks, ignore_index=True).drop_duplicates()

candidates = candidates.merge(merch_by_cmp, on='n_cmp_client', how='left')
candidates = candidates.merge(term_by_cmp, on='n_cmp_client', how='left')
candidates = candidates.merge(trx_by_agr, on='n_agr', how='left')

candidates['has_retl'] = candidates['has_retl'].fillna(0).astype(int)
candidates['has_active_term_feb'] = candidates['has_active_term_feb'].fillna(0).astype(int)
candidates['has_trx_feb_dag'] = candidates['has_trx_feb_dag'].fillna(0).astype(int)

display(candidates.head(50))

## 4) Финальная классификация причин

In [ ]:
result = only_excel_df.copy()
if len(candidates) > 0:
    result = result.merge(
        candidates[['agr_id','has_retl','has_active_term_feb','has_trx_feb_dag']],
        on='agr_id',
        how='left'
    )
else:
    result['has_retl'] = None
    result['has_active_term_feb'] = None
    result['has_trx_feb_dag'] = None

result['has_retl'] = result['has_retl'].fillna(0).astype(int)
result['has_active_term_feb'] = result['has_active_term_feb'].fillna(0).astype(int)
result['has_trx_feb_dag'] = result['has_trx_feb_dag'].fillna(0).astype(int)

def reason(row):
    if not row['in_agreements']:
        return '01_not_found_in_agreements'
    if not row['is_sa']:
        return '02_acq_class_not_SA'
    if not row['active_on_month_start']:
        return '03_inactive_on_month_start'
    if row['has_retl'] == 0:
        return '04_no_retail_points_in_ods_alpha_merchants'
    if row['has_active_term_feb'] == 0:
        return '05_no_active_terminals_in_feb'
    if row['has_trx_feb_dag'] == 0:
        return '06_no_transactions_in_feb_dag_perimeter'
    return '07_in_scope_but_missing_in_snapshot'

result['reason'] = result.apply(reason, axis=1)

reason_summary = (
    result.groupby('reason', as_index=False)
    .agg(cnt_agr=('agr_id', 'count'))
    .sort_values('cnt_agr', ascending=False)
)

display(reason_summary)
display(result.head(200))

## 5) Топ-примеры по каждой причине

In [ ]:
for r in reason_summary['reason'].tolist():
    print('\n====', r, '====')
    display(result[result['reason'] == r][['agr_id','inn','c_agr_number','acq_class','d_valid_from','d_valid_to','has_retl','has_active_term_feb','has_trx_feb_dag']].head(50))

## 6) (Опционально) экспорт результатов в Excel

In [ ]:
# result.to_excel('/home/jovyan/documents/Equaring/Проверки/only_excel_clients_reasons_full.xlsx', index=False)
# reason_summary.to_excel('/home/jovyan/documents/Equaring/Проверки/only_excel_clients_reasons_summary.xlsx', index=False)
print('Done')